# 📊 Preliminary Data Analysis — Obesity Risk Dataset

**Objectif :** Explorer le dataset pour répondre aux 4 questions critiques du projet :
1. Valeurs manquantes
2. Outliers
3. Déséquilibre des classes
4. Corrélations entre features

**Dataset :** Estimation of Obesity Levels Based on Eating Habits and Physical Condition  
**Source :** UCI Machine Learning Repository (ID: 544)  
**Cible à prédire :** `NObeyesdad` — niveau d'obésité (7 classes)

## 0. 📦 Importation des bibliothèques

On commence par importer toutes les bibliothèques nécessaires :
- **pandas** : manipulation et analyse des données (tableaux, statistiques)
- **matplotlib** : création de graphiques de base
- **seaborn** : graphiques statistiques plus avancés (heatmap, boxplot...)
- **ucimlrepo** : téléchargement direct du dataset depuis UCI

In [ ]:
# Installation de la bibliothèque UCI (nécessaire sur Google Colab)
# Le ! permet d'exécuter une commande système depuis le notebook
!pip install ucimlrepo -q

# Importation des bibliothèques
import pandas as pd                      # Manipulation des données
import matplotlib.pyplot as plt          # Graphiques de base
import seaborn as sns                    # Graphiques statistiques avancés
from ucimlrepo import fetch_ucirepo      # Téléchargement du dataset UCI

print('✅ Bibliothèques importées avec succès')

## 1. 📥 Chargement du Dataset

On télécharge le dataset directement depuis UCI avec son identifiant (id=544).  
Le dataset contient des données sur les habitudes alimentaires et conditions physiques de patients.

- `dataset.data.features` → les colonnes d'entrée (ce qu'on donne au modèle)
- `dataset.data.targets` → la colonne de sortie (ce qu'on veut prédire)
- `pd.concat(..., axis=1)` → on colle les deux côte à côte pour avoir un seul tableau

In [ ]:
# Téléchargement du dataset depuis UCI (id=544)
dataset = fetch_ucirepo(id=544)

# Fusion des features (X) et de la cible (y) en un seul DataFrame
# axis=1 signifie qu'on colle les colonnes côte à côte (pas les lignes)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

# Affichage des 5 premières lignes pour vérifier
print(f'✅ Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes')
df.head()

### 🔍 Aperçu général du dataset

`df.info()` nous donne :
- Le **type de chaque colonne** : `object` = texte, `float64` = nombre décimal
- Le **nombre de valeurs non-nulles** par colonne (utile pour détecter les valeurs manquantes)
- La **mémoire utilisée** par le dataset

In [ ]:
# Informations générales sur le dataset
# Montre les types de données et le nombre de valeurs non-nulles
df.info()

In [ ]:
# Statistiques descriptives pour les colonnes numériques
# count=nombre de valeurs, mean=moyenne, std=écart-type, min/max=extremes
# 25%/50%/75% = quartiles (50% = médiane)
df.describe()

---
## ❓ Question 1 : Valeurs Manquantes

**Pourquoi c'est important ?**  
Les modèles ML ne savent pas gérer les valeurs manquantes (NaN). Si on en a, il faut décider :
- Les **supprimer** (si peu de lignes concernées)
- Les **remplacer** par la moyenne, médiane, ou valeur la plus fréquente

`df.isnull()` crée un tableau de True/False (True = valeur manquante)  
`.sum()` compte le nombre de True par colonne

In [ ]:
# Comptage des valeurs manquantes par colonne
# isnull() retourne True là où il y a un NaN, sum() compte les True
missing_values = df.isnull().sum()

print('=== Valeurs manquantes par colonne ===')
print(missing_values)
print(f'\nTotal valeurs manquantes : {missing_values.sum()}')

### ✅ Conclusion — Valeurs Manquantes

**Résultat :** Aucune valeur manquante détectée dans le dataset (0 NaN dans toutes les colonnes).

**Stratégie :** Aucun traitement nécessaire. Le dataset est complet et prêt à l'emploi.

---
## ❓ Question 2 : Outliers (Valeurs Aberrantes)

**Qu'est-ce qu'un outlier ?**  
C'est une valeur très éloignée des autres. Par exemple, un poids de 500kg serait un outlier.

**Comment les détecter ?**  
On utilise des **boxplots** (boîtes à moustaches) :
- La boîte = 50% des données (entre Q1 et Q3)
- La ligne orange = médiane (Q2)
- Les moustaches = valeurs "normales"
- Les points isolés au-delà des moustaches = outliers potentiels

**Stratégie pour ce projet :**  
Dans un contexte médical, les valeurs extrêmes sont souvent réelles (obésité sévère, grande taille...).  
On les conserve sauf si elles sont clairement impossibles.

In [ ]:
# Liste des colonnes numériques à analyser
num_cols = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

# Création d'une grille de 2 lignes x 4 colonnes pour afficher 8 boxplots
# figsize=(14,8) définit la taille totale de la figure en pouces
fig, axes = plt.subplots(2, 4, figsize=(14, 8))

# flatten() transforme la grille 2D en liste 1D pour faciliter l'itération
axes = axes.flatten()

# Boucle sur chaque colonne numérique
for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col])          # Boxplot de la colonne
    axes[i].set_title(col)            # Titre = nom de la colonne
    axes[i].set_xlabel('')            # Supprime l'étiquette x inutile

plt.suptitle('Détection des Outliers par Variable', fontsize=14, fontweight='bold')
plt.tight_layout()   # Ajuste automatiquement les espaces entre les graphiques
plt.show()

In [ ]:
# Quantification des outliers avec la méthode IQR (Interquartile Range)
# Un outlier = valeur en dehors de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
# C'est la méthode statistique standard pour détecter les outliers

print('=== Nombre d\'outliers par colonne (méthode IQR) ===')
for col in num_cols:
    Q1 = df[col].quantile(0.25)   # 1er quartile (25% des données en dessous)
    Q3 = df[col].quantile(0.75)   # 3ème quartile (75% des données en dessous)
    IQR = Q3 - Q1                 # Écart interquartile
    
    # Bornes en dehors desquelles une valeur est considérée outlier
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    # Compte les valeurs hors bornes
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f'{col:10} : {n_outliers} outliers  (borne basse={lower:.2f}, borne haute={upper:.2f})')

### ✅ Conclusion — Outliers

**Résultat :**
- **Age** : quelques personnes de 50-60 ans → médicalement normaux, on conserve
- **Weight** : un cas à ~175kg → cas d'obésité sévère, cohérent avec le sujet, on conserve  
- **Height** : une personne à ~1.98m → plausible, on conserve
- **Autres colonnes** : outliers mineurs sur des variables discrètes (NCP, CH2O...)

**Stratégie :** On conserve tous les outliers. Dans un contexte médical, les valeurs extrêmes représentent des cas réels (obésité sévère, âge avancé) et sont importantes pour la prédiction. Les supprimer biaiserait le modèle.

---
## ❓ Question 3 : Déséquilibre des Classes

**Pourquoi c'est important ?**  
Si une classe a beaucoup plus d'exemples que les autres, le modèle apprend à toujours prédire cette classe (il sera précis mais inutile).

**Exemple de déséquilibre :** 90% Normal / 10% Obèse → le modèle dit toujours "Normal" et atteint 90% d'accuracy sans rien apprendre.

**Les 7 classes de notre dataset :**
- Insufficient_Weight
- Normal_Weight  
- Overweight_Level_I
- Overweight_Level_II
- Obesity_Type_I
- Obesity_Type_II
- Obesity_Type_III

In [ ]:
# Comptage du nombre d'exemples par classe
class_counts = df['NObeyesdad'].value_counts()

# Calcul du pourcentage par classe
# normalize=True retourne les proportions (entre 0 et 1)
# mul(100) multiplie par 100 pour avoir des pourcentages
# round(1) arrondit à 1 décimale
class_pct = df['NObeyesdad'].value_counts(normalize=True).mul(100).round(1)

print('=== Distribution des classes ===')
print('Comptage :')
print(class_counts)
print('\nPourcentages :')
print(class_pct)

In [ ]:
# Visualisation de la distribution des classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Barplot du nombre d'exemples
class_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Distribution des classes (comptage)', fontweight='bold')
axes[0].set_xlabel('Niveau d\'obésité')
axes[0].set_ylabel('Nombre d\'exemples')
axes[0].tick_params(axis='x', rotation=45)   # Rotation des labels pour lisibilité

# Graphique 2 : Pie chart des pourcentages
# autopct='%.1f%%' affiche le pourcentage avec 1 décimale sur chaque part
class_counts.plot(kind='pie', ax=axes[1], autopct='%.1f%%', startangle=90)
axes[1].set_title('Distribution des classes (pourcentages)', fontweight='bold')
axes[1].set_ylabel('')   # Supprime le label y automatique

plt.tight_layout()
plt.show()

### ✅ Conclusion — Déséquilibre des Classes

**Résultat :** Le dataset est **bien équilibré** :

| Classe | Count | % |
|--------|-------|---|
| Obesity_Type_I | 351 | 16.6% |
| Obesity_Type_III | 324 | 15.3% |
| Obesity_Type_II | 297 | 14.1% |
| Overweight_Level_I | 290 | 13.7% |
| Overweight_Level_II | 290 | 13.7% |
| Normal_Weight | 287 | 13.6% |
| Insufficient_Weight | 272 | 12.9% |

La différence entre la classe la plus représentée (16.6%) et la moins représentée (12.9%) est de seulement **3.7%** — négligeable.

**Stratégie choisie :** Utiliser `class_weight='balanced'` dans les modèles ML.  
Ce paramètre ajuste automatiquement les poids des classes pour compenser les légères différences, sans nécessiter d'oversampling (SMOTE) ou undersampling.

---
## ❓ Question 4 : Corrélations entre Features

**Pourquoi c'est important ?**  
Si deux features sont très corrélées (ex: 0.95), elles portent la même information. On peut en supprimer une pour simplifier le modèle sans perdre de performance.

**Interprétation du coefficient de corrélation :**
- **1.0** = corrélation parfaite positive (quand l'un monte, l'autre monte)
- **-1.0** = corrélation parfaite négative (quand l'un monte, l'autre descend)
- **0.0** = aucune corrélation
- **Seuil critique : > 0.8 ou < -0.8** → à surveiller

On utilise une **heatmap** (carte de chaleur) pour visualiser toutes les corrélations d'un coup.

In [ ]:
# Calcul de la matrice de corrélation
# .corr() calcule le coefficient de Pearson entre toutes les paires de colonnes numériques
corr_matrix = df[num_cols].corr()

# Visualisation avec une heatmap
plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    annot=True,        # Affiche les valeurs dans chaque case
    fmt='.2f',         # Format : 2 décimales
    cmap='coolwarm',   # Couleurs : bleu (négatif) → blanc (0) → rouge (positif)
    square=True,       # Cases carrées
    vmin=-1, vmax=1,   # Échelle fixe entre -1 et 1
    linewidths=0.5     # Lignes séparatrices entre cases
)

plt.title('Matrice de Corrélation — Features Numériques', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Identification des paires fortement corrélées (seuil > 0.5 en valeur absolue)
# On exclut la diagonale (corrélation d'une variable avec elle-même = toujours 1.0)

print('=== Paires de features corrélées (|r| > 0.5) ===')
threshold = 0.5
found = False

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):   # i+1 pour éviter les doublons
        val = corr_matrix.iloc[i, j]
        if abs(val) > threshold:   # abs() = valeur absolue (ignore le signe)
            col1 = corr_matrix.columns[i]
            col2 = corr_matrix.columns[j]
            print(f'{col1} ↔ {col2} : r = {val:.2f}')
            found = True

if not found:
    print('Aucune paire avec |r| > 0.5')

### ✅ Conclusion — Corrélations

**Résultat :** Une seule corrélation notable :
- **Height ↔ Weight = 0.46** — corrélation modérée (les personnes plus grandes pèsent généralement plus)

Toutes les autres corrélations sont inférieures à 0.30 → pas de problème de multicolinéarité.

**Stratégie :** On **conserve toutes les features**. La corrélation Height/Weight (0.46) est en dessous du seuil critique de 0.80. De plus, dans un contexte médical, les deux variables ont une signification clinique distincte : la taille seule et le poids seul sont tous deux importants pour évaluer l'obésité (cf. calcul de l'IMC = poids/taille²).

---
## 📋 Résumé Final — Preliminary Data Analysis

| Question | Résultat | Stratégie |
|----------|----------|----------|
| **Valeurs manquantes** | ✅ Aucune (0 NaN) | Aucun traitement nécessaire |
| **Outliers** | ⚠️ Quelques cas extrêmes (Age ~60, Weight ~175kg) | Conservation — médicalement plausibles |
| **Déséquilibre classes** | ✅ Bien équilibré (12.9% – 16.6%) | `class_weight='balanced'` dans les modèles |
| **Corrélations** | ⚠️ Height ↔ Weight = 0.46 (modérée) | Conservation des deux features |

**→ Le dataset est de bonne qualité et prêt pour l'entraînement des modèles ML.**